# Aperçu des données de trafic routier (octobre 2025 à septembre 2026) et du référentiel

Objectif : exposer l'état brut des deux jeux de données, avec l'ampleur de chaque problème, pour décider du nettoyage.
Rien n'est corrigé ici. Les chiffres portent sur **les 12 mois téléchargés (octobre 2025 à septembre 2026)**.

Plan :
1. Vue d'ensemble des deux fichiers
2. Référentiel : doublons, dates de validité, géométrie
3. Jointure trafic / référentiel
4. Trafic : heures manquantes
5. Trafic : valeurs manquantes (`q`, `k`)
6. Trafic : valeurs observées (zéros, extrêmes, cohérence, profil horaire)
7. Bilan et choix à faire
8. Pour le plaisir : la place de l'Étoile

In [ ]:
from itertools import combinations
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

DOSSIER = Path("../data/raw/trafic")

BLEU, ORANGE, VERT, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#a8a7a2"
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e5e1",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 9,
    "font.size": 9,
})
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 160


def nb(n):
    """Entier avec espaces comme séparateur de milliers."""
    return f"{n:,}".replace(",", " ")


def part(n, total):
    """'n (x,x %)' pour affichage."""
    p = n / total
    return f"{nb(n)} ({p:.2%} )".replace(" )", ")").replace(".", ",") if p < 0.01 else f"{nb(n)} ({p:.1%})".replace(".", ",")

## 1. Vue d'ensemble

In [ ]:
fichiers = sorted(DOSSIER.glob("trafic_*.parquet"))
trafic = pd.concat([pd.read_parquet(f) for f in fichiers], ignore_index=True)
ref = pd.read_parquet(DOSSIER / "referentiel.parquet")
DEBUT, FIN = trafic["t_1h"].min(), trafic["t_1h"].max()

pd.DataFrame({
    f"trafic ({len(fichiers)} mois)": [len(trafic), trafic.shape[1], trafic["iu_ac"].nunique(),
                                       f"{DEBUT:%Y-%m-%d %H:%M} -> {FIN:%Y-%m-%d %H:%M}",
                                       f"{trafic.memory_usage(deep=True).sum() / 1e6:,.0f} Mo".replace(",", " ")],
    "référentiel": [len(ref), ref.shape[1], ref["iu_ac"].nunique(), "-",
                    f"{ref.memory_usage(deep=True).sum() / 1e6:.1f} Mo"],
}, index=["lignes", "colonnes", "arcs distincts (iu_ac)", "période (UTC)", "mémoire"])

In [ ]:
display(trafic.head(3))
print(trafic.dtypes.to_string())

Colonnes du trafic : `iu_ac` (identifiant d'arc, texte), `t_1h` (heure UTC, **fin** de la période d'une heure), `q` (débit, véh/h), `k` (taux d'occupation, %), `etat_trafic`, `etat_barre`.
Le référentiel ne contient que de l'identification et de la géométrie, pas de mesure.

### Mois par mois

Avant de regrouper les 12 mois, on vérifie qu'ils sont comparables : un mois très différent des autres ne devrait pas être noyé dans les moyennes.

In [ ]:
mois = trafic["t_1h"].dt.strftime("%Y-%m")
par_mois = trafic.groupby(mois).agg(lignes=("iu_ac", "size"), arcs=("iu_ac", "nunique"), heures=("t_1h", "nunique"))
par_mois["heures_du_mois"] = [pd.Period(m).days_in_month * 24 for m in par_mois.index]
par_mois["heures_absentes"] = par_mois["heures_du_mois"] - par_mois["heures"]
par_mois["ni_q_ni_k"] = (trafic["q"].isna() & trafic["k"].isna()).groupby(mois).mean().mul(100).round(1)
par_mois.index.name = "mois"
par_mois

Les 12 mois ont presque le même nombre d'arcs (2 977 à 2 979) et une part de lignes sans mesure comparable (31,6 % en septembre à 35,7 % en février) : on peut les regrouper. Les heures absentes viennent de journées entières, analysées en section 4.

## 2. Référentiel

### 2.1 Un même `iu_ac` apparaît plusieurs fois

In [ ]:
n_versions = ref.groupby("iu_ac").size()
print("Lignes :", nb(len(ref)), "| arcs distincts :", nb(len(n_versions)),
      "| lignes en trop :", nb(len(ref) - len(n_versions)))
print("\nNombre de lignes par iu_ac :")
print(n_versions.value_counts().sort_index().rename("nb d'arcs").to_string())

cols = ["iu_ac", "libelle", "date_debut", "date_fin", "iu_nd_amont", "iu_nd_aval"]
display(ref[ref["iu_ac"] == "1027"][cols])

In [ ]:
# Les coordonnées sont en WGS84 (longitude / latitude, EPSG:4326) ; on passe en
# Lambert-93 (EPSG:2154) pour mesurer en mètres. geo_shape devient la colonne géométrie,
# on retire les colonnes WKB brutes devenues inutiles.
ref_geo = gpd.GeoDataFrame(ref.drop(columns=["geo_shape", "geo_point_2d"]),
                           geometry=gpd.GeoSeries.from_wkb(ref["geo_shape"], crs=4326).to_crs(2154))
doublons = ref_geo[ref_geo["iu_ac"].duplicated(keep=False)]


def ecarts_versions(g):
    """Écart max entre deux versions d'un arc, toutes paires comparées (m)."""
    paires = list(combinations(g.geometry, 2))
    return pd.Series({
        "ecart_centres": max(a.centroid.distance(b.centroid) for a, b in paires),
        "ecart_hausdorff": max(a.hausdorff_distance(b) for a, b in paires),
        "ecart_longueur": g.length.max() - g.length.min(),
    })


ecarts = doublons.groupby("iu_ac").apply(ecarts_versions, include_groups=False)
print(f"{len(ecarts)} arcs ont plusieurs lignes ; écarts entre versions (m) :")
display(ecarts.describe(percentiles=[.5, .9]).round(0))

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(ecarts["ecart_hausdorff"], bins=30, color=BLEU)
ax.set_title(f"Deux versions d'un même arc diffèrent de {ecarts['ecart_hausdorff'].median():.0f} m en médiane")
ax.set_xlabel("distance de Hausdorff maximale entre versions (m)")
ax.set_ylabel("nombre d'arcs")
plt.show()

Les doublons ne sont pas des copies : les 338 arcs concernés ont des **géométries différentes**. Le centre des versions bouge de 112 m en médiane, mais ce chiffre mélange déplacement et changement de longueur (un tracé plus long ou plus court déplace son centre) ; la distance de Hausdorff (écart le plus grand entre deux tracés) est plus fidèle : 167 m en médiane. Nous ne savons pas encore si ce sont des tracés successifs ou des tronçons différents du même axe. Il faudra une règle pour n'en garder qu'une, sinon la jointure multiplie les lignes (voir section 3).

#### Versions impossibles à départager par les dates

Règle envisagée : garder la version dont `date_fin` est la plus récente, puis `date_debut` en cas d'égalité. Elle ne départage pas tout : pour quelques arcs, deux versions ont exactement les mêmes dates.

In [ ]:
derniere_fin = ref_geo.groupby("iu_ac")["date_fin"].transform("max")
candidats = ref_geo[ref_geo["date_fin"] == derniere_fin]
candidats = candidats[candidats["date_debut"] == candidats.groupby("iu_ac")["date_debut"].transform("max")]
indecis = candidats[candidats["iu_ac"].duplicated(keep=False)]
display(indecis[["iu_ac", "libelle", "date_debut", "date_fin", "libelle_nd_amont", "libelle_nd_aval"]])
print("Lignes de trafic pour ces arcs :", trafic["iu_ac"].isin(indecis["iu_ac"]).sum(),
      "| dont avec q ou k :", trafic[trafic["iu_ac"].isin(indecis["iu_ac"])][["q", "k"]].notna().any(axis=1).sum())

groupes = list(indecis.groupby("iu_ac"))
fig, axes = plt.subplots(1, 1 + len(groupes), figsize=(15, 5.5), gridspec_kw={"width_ratios": [1.4] + [1] * len(groupes)})

# à gauche : position des arcs dans Paris
ax = axes[0]
ref_geo.plot(ax=ax, color=GRIS, linewidth=0.4, alpha=0.6)
for (iu, g), decalage in zip(groupes, [(40, -50), (-90, 50)]):
    g.plot(ax=ax, color=ORANGE, linewidth=3)
    centre = g.geometry.union_all().centroid
    ax.annotate(f"{g['libelle'].iloc[0]} (iu_ac {iu})", xy=(centre.x, centre.y), xytext=decalage,
                textcoords="offset points", arrowprops={"arrowstyle": "-", "color": ORANGE}, fontsize=9)
ax.set_title(f"{len(groupes)} arcs ont deux versions aux dates identiques")
ax.set_axis_off()

# à droite : zoom sur chaque arc, une couleur par version
marge = 150
for zoom, (iu, g) in zip(axes[1:], groupes):
    ref_geo.plot(ax=zoom, color=GRIS, linewidth=1.5, alpha=0.6)
    for (_, version), couleur, numero in zip(g.iterrows(), [BLEU, ORANGE], [1, 2]):
        gpd.GeoSeries([version.geometry], crs=2154).plot(ax=zoom, color=couleur, linewidth=4, alpha=0.8,
                                                         label=f"version {numero} ({version.geometry.length:.0f} m)")
    xmin, ymin, xmax, ymax = g.total_bounds
    zoom.set_xlim(xmin - marge, xmax + marge)
    zoom.set_ylim(ymin - marge, ymax + marge)
    zoom.legend(frameon=False, loc="lower left", fontsize=8)
    zoom.set_title(f"{g['libelle'].iloc[0]} (iu_ac {iu}) : écart de {ecarts.loc[iu, 'ecart_hausdorff']:.0f} m")
    zoom.set_axis_off()
plt.show()

Les deux versions de chaque arc ne se superposent pas : ce sont deux morceaux consécutifs de la même voie (voie express rive gauche, souterrain sous l'Étoile), qui ont peut-être formé l'arc ensemble. Aucun de ces deux arcs n'a de ligne dans le trafic : le choix de la version n'a pas d'effet sur la suite.

### 2.2 Dates de validité inexploitables

In [ ]:
for c in ["date_debut", "date_fin"]:
    print(c, ":", ref[c].min().date(), "->", ref[c].max().date())

print("\nLignes dont date_fin est antérieure au début de la période :", part((ref["date_fin"] < DEBUT).sum(), len(ref)))
print("\nValeurs les plus fréquentes de date_fin :")
print(ref["date_fin"].dt.date.value_counts().head(4).to_string())

en_trafic = ref[ref["iu_ac"].isin(trafic["iu_ac"])]
print("\nParmi les lignes d'arcs présents dans le trafic, date_fin < début de la période :",
      part((en_trafic["date_fin"] < DEBUT).sum(), len(en_trafic)))

Aucune ligne du référentiel n'est « valide » sur la période (octobre 2025 - septembre 2026) d'après ses dates, y compris pour des arcs qui remontent des mesures. **On ne peut pas filtrer le référentiel sur `date_debut` / `date_fin`.**

### 2.3 Géométrie

In [ ]:
plus_long = ref_geo.loc[[ref_geo.length.idxmax()]]
# aucun arc ne s'appelle "Vavin" : on prend les arcs qui aboutissent au carrefour Bd_Raspail-Vavin
vavin = ref_geo[ref_geo["libelle_nd_amont"].str.contains("Vavin") | ref_geo["libelle_nd_aval"].str.contains("Vavin")]

fig, (ax, zoom) = plt.subplots(1, 2, figsize=(13, 6.5), gridspec_kw={"width_ratios": [1.2, 0.8]})

# à gauche : tout Paris
ref_geo.plot(ax=ax, color=GRIS, linewidth=0.4, alpha=0.6)
for arcs, couleur, texte, decalage in [
    (plus_long, ORANGE, f"{plus_long.iloc[0]['libelle']}\n(iu_ac {plus_long.iloc[0]['iu_ac']}, {plus_long.length.iloc[0]:.0f} m)", (40, 40)),
    (vavin, BLEU, f"{vavin.iloc[0]['libelle']} au carrefour Vavin\n(iu_ac {', '.join(sorted(vavin['iu_ac']))})", (-130, -60)),
]:
    arcs.plot(ax=ax, color=couleur, linewidth=3)
    centre = arcs.geometry.union_all().centroid
    ax.annotate(texte, xy=(centre.x, centre.y), xytext=decalage, textcoords="offset points",
                arrowprops={"arrowstyle": "-", "color": couleur}, fontsize=9)
ax.set_title("Deux arcs repères : le plus long (orange) et le carrefour Vavin (bleu)")
ax.set_axis_off()

# à droite : zoom sur le carrefour (marge de 80 m autour des 4 arcs)
xmin, ymin, xmax, ymax = vavin.total_bounds
marge = 80
ref_geo.plot(ax=zoom, color=GRIS, linewidth=1.5, alpha=0.6)
couleurs = {"Montparnasse": BLEU, "Stanislas": VERT}
# un tronçon = les deux arcs entre Vavin et un même carrefour voisin, un par sens
voisins = (vavin["libelle_nd_amont"] + vavin["libelle_nd_aval"]).map(
    lambda t: "Montparnasse" if "Montparnasse" in t else "Stanislas")
x_milieu = vavin.geometry.map(lambda g: g.interpolate(0.5, normalized=True).x)
x_troncon = x_milieu.groupby(voisins).transform("mean")  # x moyen des deux arcs du même tronçon
for i, arc in vavin.iterrows():
    voisin = voisins[i]
    gpd.GeoSeries([arc.geometry], crs=2154).plot(ax=zoom, color=couleurs[voisin], linewidth=4)
    debut, milieu, fin = (arc.geometry.interpolate(f, normalized=True) for f in (0.35, 0.5, 0.65))
    zoom.annotate("", xy=(fin.x, fin.y), xytext=(debut.x, debut.y),
                  arrowprops={"arrowstyle": "-|>", "color": "black", "linewidth": 1.2})
    cote = -1 if x_milieu[i] < x_troncon[i] else 1
    zoom.annotate(arc["iu_ac"], xy=(milieu.x, milieu.y), xytext=(14 * cote, 0), textcoords="offset points",
                  ha="left" if cote > 0 else "right", va="center", fontsize=9)
noeud = vavin[vavin["libelle_nd_aval"].str.contains("Vavin")].iloc[0].geometry.coords[-1]
zoom.plot(*noeud, "o", color="black", markersize=6)
zoom.annotate("carrefour Vavin", xy=noeud, xytext=(60, 12), textcoords="offset points", fontsize=9,
              arrowprops={"arrowstyle": "-", "color": "black"})
zoom.set_xlim(xmin - marge, xmax + marge)
zoom.set_ylim(ymin - marge, ymax + marge)
zoom.set_title("Zoom : 4 arcs = 2 tronçons, un arc par sens (flèches)")
zoom.set_axis_off()
plt.show()

In [ ]:
print("Types de géométrie :", ref_geo.geom_type.value_counts().to_dict())
print("Géométries invalides ou vides :", int((~ref_geo.is_valid).sum() + ref_geo.is_empty.sum()))
lon_lat = ref_geo.geometry.centroid.to_crs(4326)
print(f"Longitude : {lon_lat.x.min():.3f} -> {lon_lat.x.max():.3f} | latitude : {lon_lat.y.min():.3f} -> {lon_lat.y.max():.3f}")
longueur = ref_geo.length
print("\nLongueur des arcs (m) :")
print(longueur.describe(percentiles=[.05, .5, .95]).round(0).to_string())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
ref_geo.plot(ax=a1, color=BLEU, linewidth=0.5)
a1.set_title("Les arcs couvrent tout Paris, périphérique compris")
a1.set_axis_off()
a2.hist(longueur, bins=40, color=BLEU)
a2.set_title(f"Un arc mesure {longueur.median():.0f} m en médiane")
a2.set_xlabel("longueur de l'arc (m)")
a2.set_ylabel("nombre de lignes du référentiel")
plt.show()

## 3. Jointure trafic / référentiel (clé `iu_ac`)

In [ ]:
ids_trafic, ids_ref = set(trafic["iu_ac"]), set(ref["iu_ac"])
sans_ref = trafic[~trafic["iu_ac"].isin(ids_ref)]
a_valeur = sans_ref[sans_ref[["q", "k"]].notna().any(axis=1)]

print("Arcs dans le trafic :", nb(len(ids_trafic)), "| dans le référentiel :", nb(len(ids_ref)),
      "| communs :", nb(len(ids_trafic & ids_ref)))
print("\nArcs du trafic absents du référentiel :", len(ids_trafic - ids_ref),
      "->", part(len(sans_ref), len(trafic)), "des lignes")
print("  dont lignes avec au moins une mesure (q ou k) :", nb(len(a_valeur)),
      "sur", len(a_valeur["iu_ac"].unique()), "arcs")
print("Arcs du référentiel sans aucune ligne dans le trafic :", len(ids_ref - ids_trafic))

jointure = trafic.merge(ref[["iu_ac"]], on="iu_ac", how="left")
print("\nJointure directe, sans dédoublonner le référentiel :")
print("  lignes avant :", nb(len(trafic)), "| après :", nb(len(jointure)),
      f"(+{len(jointure) / len(trafic) - 1:.1%})".replace(".", ","))
print("  part des lignes de trafic dont l'arc a plusieurs versions :",
      f"{trafic['iu_ac'].isin(n_versions[n_versions > 1].index).mean():.1%}".replace(".", ","))

Deux points à retenir : (1) 31 arcs du trafic n'ont **pas de géométrie** (donc pas de rattachement possible à une station) ; 21 d'entre eux ont des mesures, soit 149 675 lignes utiles perdues ; (2) sans dédoublonnage, la jointure gonfle le trafic de 12 % (3,2 millions de lignes en double).

### Les 29 arcs sans référentiel : leur nom existe-t-il quand même ?

Le fichier de trafic téléchargé ne garde que 6 colonnes. Le jeu de l'API contient aussi `libelle`, les nœuds amont / aval et la géométrie : on interroge l'API sur toute la période pour ces arcs, en regroupant par arc (une ligne par arc, pas par heure). Requête d'exploration seulement, rien n'est enregistré.

In [ ]:
import requests

URL_API = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/comptages-routiers-permanents/records"
ids_sans_ref = sorted(ids_trafic - ids_ref, key=int)
ids = ",".join(f"'{a}'" for a in ids_sans_ref)
periode_api = f"t_1h >= date'{DEBUT:%Y-%m-%d}' and t_1h < date'{FIN + pd.Timedelta(days=1):%Y-%m-%d}'"


def requete_api(select, group_by, complement=""):
    """Une ligne par groupe pour les arcs sans référentiel, sur toute la période."""
    rep = requests.get(URL_API, timeout=120, params={
        "select": select, "group_by": group_by,
        "where": f"iu_ac in ({ids}) and {periode_api}{complement}", "limit": 100})
    rep.raise_for_status()
    return pd.DataFrame(rep.json()["results"])


champs_noms = "iu_ac,libelle,libelle_nd_amont,libelle_nd_aval"
api = requete_api(champs_noms, champs_noms).drop_duplicates("iu_ac")
print(len(api), "arcs retrouvés sur", len(ids_sans_ref))
# nombre d'arcs dont le champ est renseigné au moins une fois
non_vides = {champ: len(requete_api("iu_ac,count(*) as n", "iu_ac", f" and {champ} is not null"))
             for champ in ["geo_point_2d", "geo_shape", "date_debut"]}
print("Arcs avec un champ renseigné :", {"libelle": int(api["libelle"].notna().sum()), **non_vides})

mesures = trafic[trafic["iu_ac"].isin(api["iu_ac"])].groupby("iu_ac").agg(h_q=("q", "count"), h_k=("k", "count"))
sans_ref_noms = (api[["iu_ac", "libelle", "libelle_nd_amont", "libelle_nd_aval"]]
                 .merge(mesures, left_on="iu_ac", right_index=True))
sans_ref_noms["avec_mesures"] = (sans_ref_noms["h_q"] + sans_ref_noms["h_k"]) > 0
sans_ref_noms = (sans_ref_noms.sort_values(["avec_mesures", "iu_ac"], ascending=[False, True],
                                           key=lambda s: s.astype(int) if s.name == "iu_ac" else s)
                 .set_index("iu_ac"))
print(f"Avec au moins une mesure : {sans_ref_noms['avec_mesures'].sum()} | sans aucune mesure : {(~sans_ref_noms['avec_mesures']).sum()}")
sans_ref_noms

Les 31 arcs ont un nom (`libelle`) et des nœuds amont / aval dans l'API, mais **aucun centre ni tracé** (`geo_point_2d` et `geo_shape` sont vides, comme `date_debut`). Sans position, ils ne peuvent pas être rattachés à une station par un rayon ; leur nom ne permet que de les situer à la main. Plusieurs libellés sont des codes techniques (`CF1424->CF0181`, `Place Dauphine->CF1994`, `1711->4`). Par rapport à octobre seul (29 arcs), deux arcs s'ajoutent : 7290 et 7291, qui n'apparaissent qu'à partir de mai et juillet 2026.

## 4. Trafic : heures manquantes

Grille complète = tous les arcs × toutes les heures de la période.

In [ ]:
heures = pd.date_range(DEBUT, FIN, freq="h")
arcs = trafic["iu_ac"].unique()
attendues = len(heures) * len(arcs)
print("Heures :", nb(len(heures)), "| arcs :", len(arcs), "| lignes attendues :", nb(attendues))
print("Lignes présentes :", nb(len(trafic)), "| manquantes :", part(attendues - len(trafic), attendues))
print("Doublons exacts :", trafic.duplicated().sum(), "| doublons (iu_ac, t_1h) :", trafic.duplicated(["iu_ac", "t_1h"]).sum())

# grille restreinte à la vie de chaque arc (de sa première à sa dernière heure)
vie = trafic.groupby("iu_ac")["t_1h"].agg(premiere="min", derniere="max", lignes="size")
vie["attendues"] = ((vie["derniere"] - vie["premiere"]) / pd.Timedelta(hours=1)).astype(int) + 1
print("Manquantes entre la première et la dernière heure de chaque arc :",
      part(int(vie["attendues"].sum() - vie["lignes"].sum()), int(vie["attendues"].sum())))

lignes_par_heure = trafic.groupby("t_1h").size().reindex(heures, fill_value=0)
heure_vide = lignes_par_heure[lignes_par_heure == 0].index.to_series()
print("\nHeures sans aucune ligne :", len(heure_vide), "sur", nb(len(heures)))
blocs = heure_vide.groupby((heure_vide.diff() != pd.Timedelta(hours=1)).cumsum().values).agg(debut="min", fin="max", heures="size")
blocs["debut"], blocs["fin"] = blocs["debut"].dt.strftime("%Y-%m-%d %H:%M"), blocs["fin"].dt.strftime("%Y-%m-%d %H:%M")
display(blocs.reset_index(drop=True))

seuil_partiel = 0.9 * lignes_par_heure.median()
partielles = lignes_par_heure[(lignes_par_heure > 0) & (lignes_par_heure < seuil_partiel)]
print(f"Heures présentes mais avec moins de 90 % des lignes habituelles : {len(partielles)}")

In [ ]:
libelle = ref.drop_duplicates("iu_ac").set_index("iu_ac")["libelle"].combine_first(sans_ref_noms["libelle"])

# arcs qui apparaissent après le début ou disparaissent avant la fin de la période
partiels = vie[(vie["premiere"] > DEBUT) | (vie["derniere"] < FIN)].copy()
sous = trafic[trafic["iu_ac"].isin(partiels.index)]
partiels["libelle"] = partiels.index.map(libelle)
partiels["part_q_nan"] = sous.groupby("iu_ac")["q"].apply(lambda s: s.isna().mean()).round(2)
partiels["part_barre"] = sous.groupby("iu_ac")["etat_barre"].apply(lambda s: (s == "Barré").mean()).round(2)
print(f"{len(partiels)} arcs ne couvrent pas toute la période :")
display(partiels.sort_values("premiere")[["libelle", "premiere", "derniere", "lignes", "part_q_nan", "part_barre"]])

# les arcs Barré disparaissent-ils en général ?
h_barre = trafic[trafic["etat_barre"] == "Barré"].groupby("iu_ac").size()
print(f"Arcs avec au moins une heure Barré : {len(h_barre)} | Barré pendant toute leur présence : {(h_barre == vie.loc[h_barre.index, 'lignes']).sum()}")

Le trou est faible (1,6 % de la grille complète, 1,5 % si on ne compte que la vie de chaque arc) et se décompose ainsi :
- **129 heures sans aucune ligne, pour tous les arcs à la fois**, en 11 blocs : cinq journées de 24 h (29 décembre, 29 janvier, 16 février, 24 mars, du 12 au 13 avril) et six blocs courts de 1 à 3 h. Aucune heure n'est « partielle » : une heure est soit complète, soit absente. Ce sont des interruptions de la collecte, pas des pannes de capteurs ;
- l'heure du **2025-10-26 00:00 UTC** coïncide avec le changement d'heure d'automne. Il n'y a en revanche **aucun trou au changement d'heure de mars 2026**, ce qui n'est cohérent qu'avec une cause propre à l'heure d'automne (heure locale répétée, probablement écartée à l'export). À vérifier au nettoyage ;
- **6 arcs ne couvrent pas toute la période**. Deux arcs de l'avenue Claude-Vellefaux (4979, 4981) s'arrêtent le 22 juillet 2026 alors qu'ils mesuraient (5 % de `q` manquant) ; l'arc 1005 (Alésia) s'arrête le 9 juin 2026 et l'arc 233 (Concorde_Est) le 13 octobre 2025, tous deux sans aucune mesure `q` ; les arcs 7290 et 7291 apparaissent en mai et juillet 2026 et ne sont pas dans le référentiel. On ne sait pas pourquoi ces arcs entrent ou sortent du jeu (retrait, travaux, nouveaux capteurs) ;
- 1 767 arcs ont au moins une heure `Barré`, mais un seul (l'arc 233) l'est pendant toute sa présence : `Barré` n'annonce pas la disparition d'un arc.

La grille horaire complète sera nécessaire pour rendre ces trous visibles.

### Disponibilité de `q` et `k` au fil de l'année

Part des couples (arc, heure) de la grille complète où la mesure est renseignée, jour par jour, puis arc par arc. Une heure sans aucune ligne compte comme non renseignée.

In [ ]:
# disponibilité par jour : part des couples (arc, heure) de la grille complète où q (ou k) est renseigné
jours = pd.date_range(DEBUT.floor("D"), FIN.floor("D"), freq="D", tz="UTC")
jour_ligne = trafic["t_1h"].dt.floor("D")
par_jour = pd.DataFrame({c: trafic[c].notna().groupby(jour_ligne).sum() for c in ["q", "k"]})
par_jour = par_jour.reindex(jours, fill_value=0) / (24 * len(arcs))
h_absentes = heure_vide.dt.floor("D").value_counts().reindex(jours, fill_value=0)

MOIS = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]
debuts_mois = jours[jours.day == 1]
x_jours = np.arange(len(jours))
x_mois = [x_jours[jours.get_loc(d)] for d in debuts_mois]
lib_mois = [f"{MOIS[d.month - 1]} {d.year % 100}" for d in debuts_mois]

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
a1.plot(x_jours, par_jour["q"] * 100, color=BLEU, linewidth=1.5, label="q (débit)")
a1.plot(x_jours, par_jour["k"] * 100, color=ORANGE, linewidth=1.5, label="k (taux d'occupation)")
a1.set_ylim(0, 100)
a1.set_ylabel("couples (arc, heure)\nrenseignés (%)")
mq, mk = par_jour["q"].median(), par_jour["k"].median()
a1.set_title(f"Environ {mq:.0%} (q) et {mk:.0%} (k) des couples arc-heure sont renseignés chaque jour, sauf pendant les coupures"
             .replace(".", ","))
a1.legend(frameon=False, loc="lower left", ncol=2)

a2.bar(x_jours, h_absentes, width=1, color=GRIS)
a2.set_ylabel("heures sans\naucune ligne")
a2.set_ylim(0, 24)
a2.set_yticks([0, 12, 24])
a2.set_xticks(x_mois[::2], lib_mois[::2])
a2.set_xlim(-1, len(jours))
a2.set_xlabel("jour (UTC)")
plt.tight_layout()
plt.show()

creux = par_jour[(par_jour["q"] < 0.9 * mq) & (h_absentes == 0)]
print(f"Jours sans heure absente où q passe sous 90 % de la médiane : {len(creux)} ({', '.join(creux.index.strftime('%d/%m'))})")
print("Jours avec au moins une heure absente :", int((h_absentes > 0).sum()), "| heures absentes :", int(h_absentes.sum()))

- **Le niveau est stable** : environ 54 % des couples (arc, heure) ont `q` et 57 % ont `k`, tous les jours de l'année, avec un écart constant entre les deux courbes. Le plafond n'est pas 100 % parce que de nombreux arcs ne mesurent jamais (voir plus bas).
- **Les chutes à zéro sont les coupures de collecte** de la section précédente (29 décembre, 29 janvier, 16 février, 24 mars, 12-13 avril) : elles touchent tous les arcs en même temps.
- **Huit autres jours passent sous 90 % de la médiane sans heure absente** : du 9 au 15 février (hors le 10) et les 21 et 22 juin. Les lignes sont là mais les mesures manquent : c'est une panne d'une partie des capteurs, pas de la collecte.

In [ ]:
# disponibilité par arc et par jour : part des 24 heures du jour où la mesure est renseignée
m_jour = {}
for col in ["q", "k"]:
    m = trafic[col].notna().groupby([trafic["iu_ac"], jour_ligne]).sum().unstack(fill_value=0)
    m_jour[col] = m.reindex(columns=jours, fill_value=0) / 24
ordre = (m_jour["q"] + m_jour["k"]).mean(axis=1).sort_values(ascending=False).index  # même ordre pour q et k

cmap_dispo = LinearSegmentedColormap.from_list("dispo", ["#f3f2ee", "#2a78d6", "#08306b"])
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for ax, col, nom in zip(axes, ["q", "k"], ["q (débit)", "k (taux d'occupation)"]):
    img = ax.imshow(m_jour[col].loc[ordre].values, aspect="auto", cmap=cmap_dispo, vmin=0, vmax=1,
                    interpolation="nearest", extent=[-0.5, len(jours) - 0.5, len(ordre), 0])
    ax.grid(False)
    ax.set_ylabel(f"{nom}\narcs (triés par disponibilité)")
jamais_q, presque_q = (m_jour["q"].mean(axis=1) == 0).mean(), (m_jour["q"].mean(axis=1) > 0.95).mean()
axes[0].set_title(f"{jamais_q:.0%} des arcs n'ont jamais de q et {presque_q:.0%} en ont presque toujours : les autres alternent mesures et pannes"
                  .replace(".", ","))
axes[1].set_xticks(x_mois[::2], lib_mois[::2])
axes[1].set_xlabel("jour (UTC)")
fig.colorbar(img, ax=axes, shrink=0.5, pad=0.02, label="part des 24 h du jour renseignées")
plt.show()

In [ ]:
# arcs dont la mesure démarre ou s'arrête en cours de période (hors arcs jamais mesurés)
marge = pd.Timedelta(days=30)
bornes = {}
for col in ["q", "k"]:
    v = trafic[trafic[col].notna()].groupby("iu_ac")["t_1h"].agg(premiere="min", derniere="max")
    bornes[col] = pd.Series({"mesurés au moins une fois": len(v),
                             "démarrent plus de 30 j après le début": int((v["premiere"] > DEBUT + marge).sum()),
                             "s'arrêtent plus de 30 j avant la fin": int((v["derniere"] < FIN - marge).sum())})
display(pd.DataFrame(bornes))

- **La disponibilité se joue arc par arc** : 31 % des arcs n'ont jamais de `q` (28 % pour `k`), 21 % en ont presque tous les jours, les autres alternent mesures et pannes (bandes horizontales claires). Les colonnes claires verticales sont les coupures de collecte.
- **Des arcs commencent ou cessent de mesurer en cours d'année** (tableau) : environ 390 arcs commencent à mesurer plus de 30 jours après le début de la période et 250 à 300 s'arrêtent plus de 30 jours avant la fin. Ils existent dans la grille toute l'année, avec des `NaN` avant ou après.
- **Conséquence pour le nettoyage** : la part de `NaN` d'un arc sur toute la période ne distingue pas un capteur en panne de temps en temps d'un capteur qui n'existait pas encore ou qui a été retiré. Les coupures de collecte, communes à tous les arcs, gonflent aussi cette part de la même quantité pour chacun. Il faudra décider si le seuil se calcule sur la vie de chaque arc (de sa première à sa dernière mesure) et hors coupures.

## 5. Trafic : valeurs manquantes (`q` et `k`)

### 5.1 Au niveau des lignes

In [ ]:
q_nan, k_nan = trafic["q"].isna(), trafic["k"].isna()
n = len(trafic)
repartition = pd.Series({
    "q et k renseignés": (~q_nan & ~k_nan).sum(),
    "k seul (q manquant)": (q_nan & ~k_nan).sum(),
    "q seul (k manquant)": (~q_nan & k_nan).sum(),
    "ni q ni k": (q_nan & k_nan).sum(),
})
display(pd.DataFrame({"lignes": repartition.map(nb), "part": (repartition / n).map("{:.1%}".format)}))

fig, ax = plt.subplots(figsize=(7, 2.4))
couleurs = [BLEU, VERT, ORANGE, GRIS]
ax.barh(repartition.index[::-1], (repartition / n * 100)[::-1], color=couleurs[::-1])
ax.set_title(f"Seules {repartition.iloc[0] / n:.0%} des lignes ont à la fois q et k".replace(".", ","))
ax.set_xlabel("part des lignes (%)")
ax.grid(axis="y", visible=False)
plt.show()

### 5.2 Lien avec `etat_barre` et `etat_trafic`

In [ ]:
print("Part de NaN selon etat_barre :")
display(trafic.groupby("etat_barre")[["q", "k"]].agg(lambda s: s.isna().mean()).join(
    trafic["etat_barre"].value_counts().rename("lignes")).round(3))

print("Part de NaN et valeurs de k selon etat_trafic :")
display(trafic.groupby("etat_trafic").agg(lignes=("k", "size"), k_nan=("k", lambda s: s.isna().mean()),
                                          k_min=("k", "min"), k_max=("k", "max"),
                                          q_nan=("q", lambda s: s.isna().mean())).round(3))

- `Invalide` : q et k sont **toujours** NaN (28 % des lignes).
- `etat_trafic = Inconnu` correspond exactement à k NaN : cette colonne est **déduite de k** (seuils à 15 / 30 / 50 %) et n'apporte aucune information en plus.
- `Ouvert` et `Barré` n'assurent pas une mesure : il reste des NaN dans les deux cas. `etat_barre` seul ne suffit donc pas pour filtrer.

### 5.3 Au niveau des arcs : des arcs vides, et des arcs presque complets

In [ ]:
par_arc = trafic.groupby("iu_ac").agg(
    q_nan=("q", lambda s: s.isna().mean()),
    k_nan=("k", lambda s: s.isna().mean()),
    invalide=("etat_barre", lambda s: (s == "Invalide").mean()),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, col, nom in zip(axes, ["q_nan", "k_nan"], ["q (débit)", "k (taux d'occupation)"]):
    ax.hist(par_arc[col] * 100, bins=20, color=BLEU)
    ax.set_title(f"{nom} : {(par_arc[col] == 1).sum()} arcs sur {len(par_arc)} sont 100 % vides")
    ax.set_xlabel("part d'heures manquantes de l'arc (%)")
axes[0].set_ylabel("nombre d'arcs")
plt.show()

In [ ]:
seuils = [0.05, 0.10, 0.20, 0.50]
lignes = []
for s in seuils:
    q_ok, k_ok = par_arc["q_nan"] <= s, par_arc["k_nan"] <= s
    lignes.append({"part de NaN max. tolérée": f"{s:.0%}", "q": q_ok.sum(), "k": k_ok.sum(),
                   "q et k": (q_ok & k_ok).sum(), "q ou k": (q_ok | k_ok).sum()})
tableau = pd.DataFrame(lignes).set_index("part de NaN max. tolérée")
print(f"Nombre d'arcs conservés (sur {len(par_arc)}) selon le seuil de NaN toléré :")
display(tableau)

In [ ]:
# Type d'arc, avec 50 % de NaN comme limite
q_ok, k_ok = par_arc["q_nan"] <= 0.5, par_arc["k_nan"] <= 0.5
type_arc = np.select([q_ok & k_ok, k_ok, q_ok], ["q et k", "k seul", "q seul"], "vide")
type_arc = pd.Series(type_arc, index=par_arc.index, name="type")
print(type_arc.value_counts().to_string())

La distribution est **bimodale** : la plupart des arcs sont soit presque complets, soit entièrement vides, avec une queue d'arcs partiellement renseignés (pannes sur une partie de l'année). Passer d'un seuil de 5 % à 50 % ajoute environ 850 arcs pour `q` (854 -> 1 704) : sur 12 mois, le choix du seuil pèse plus qu'en octobre seul.

Avec 50 % de NaN comme limite : 1 402 arcs ont q et k, 381 seulement k, 302 seulement q, 896 sont vides. Exiger q **et** k retire donc environ 680 arcs de plus que d'exiger q **ou** k.

### 5.4 Où sont les arcs utilisables ?

In [ ]:
carte = (ref_geo.sort_values("date_fin").drop_duplicates("iu_ac", keep="last")  # une version arbitraire par arc, pour l'affichage
         .merge(type_arc, left_on="iu_ac", right_index=True, how="left"))
carte["type"] = carte["type"].fillna("absent du trafic")

style = {"q et k": (BLEU, 1.2), "k seul": (VERT, 1.0), "q seul": (ORANGE, 1.0),
         "vide": (GRIS, 0.5), "absent du trafic": ("#d8d7d2", 0.4)}
fig, ax = plt.subplots(figsize=(7.5, 6.5))
for nom in ["absent du trafic", "vide", "k seul", "q seul", "q et k"]:
    sous = carte[carte["type"] == nom]
    sous.plot(ax=ax, color=style[nom][0], linewidth=style[nom][1], label=f"{nom} ({len(sous)})")
ax.legend(title="arcs du référentiel", loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False, fontsize=8)
ax.set_title("Les arcs sans mesure sont répartis dans tout Paris")
ax.set_axis_off()
plt.show()

## 6. Trafic : valeurs observées

### 6.1 Distributions

In [ ]:
display(trafic[["q", "k"]].describe(percentiles=[.01, .5, .9, .99, .999]).T)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
a1.hist(trafic["q"].dropna(), bins=60, color=BLEU)
a1.set_yscale("log")
a1.set_title(f"q : médiane {trafic['q'].median():.0f} véh/h, 2e groupe vers 5 000")
a1.set_xlabel("q, débit (véh/h)")
a1.set_ylabel("nombre de lignes (échelle log)")
a2.hist(trafic["k"].dropna(), bins=60, color=BLEU)
a2.set_yscale("log")
a2.set_title(f"k : médiane {trafic['k'].median():.0f} %, queue jusqu'à 99 %")
a2.set_xlabel("k, taux d'occupation (%)")
plt.show()

print("Valeurs négatives : q", int((trafic["q"] < 0).sum()), "| k", int((trafic["k"] < 0).sum()),
      "| k > 100 :", int((trafic["k"] > 100).sum()))
print("Types de iu_ac :", trafic["iu_ac"].map(type).value_counts().to_dict(),
      "| tous numériques :", bool(trafic["iu_ac"].str.isnumeric().all()))

Pas de valeur négative ni de taux d'occupation > 100 % sur les 12 mois. Le débit `q` a deux régimes : la grande majorité des lignes sous 2 000 véh/h, et un second groupe autour de 4 000-6 000 véh/h (arcs à fort débit, voir 6.3). Le maximum est de 11 793 véh/h, mais seules 48 lignes dépassent 7 500 véh/h (12 dépassent 10 000) : ces valeurs isolées sont à examiner au nettoyage.

### 6.2 Zéros : vrais zéros ou pannes ?

In [ ]:
zq, zk = trafic["q"] == 0, trafic["k"] == 0
print("q = 0 :", part(zq.sum(), trafic["q"].notna().sum()), "des q renseignés")
print("k = 0 :", part(zk.sum(), trafic["k"].notna().sum()), "des k renseignés")

zeros_par_heure = trafic[zq].groupby(trafic["t_1h"].dt.hour).size().reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(zeros_par_heure.index, zeros_par_heure.values, color=BLEU)
ax.set_title("Les débits nuls se concentrent la nuit (UTC)")
ax.set_xlabel("heure UTC (fin de période)")
ax.set_ylabel("nombre de lignes avec q = 0")
plt.show()

In [ ]:
tri = trafic.sort_values(["iu_ac", "t_1h"])
tri["q_zero"] = tri["q"] == 0
zeros_arc = tri.groupby("iu_ac").agg(zeros=("q_zero", "sum"), q_renseignes=("q", "count"))
zeros_arc["part_zeros"] = (zeros_arc["zeros"] / zeros_arc["q_renseignes"]).round(3)

# plus longue suite de lignes consécutives avec q = 0, par arc (une nouvelle suite commence à chaque changement de valeur ou d'arc)
bloc = (tri["q_zero"] != tri.groupby("iu_ac")["q_zero"].shift()).cumsum()
series = tri[tri["q_zero"]].groupby(bloc[tri["q_zero"]]).agg(iu_ac=("iu_ac", "first"), heures=("q_zero", "size"))
zeros_arc["plus_longue_serie_h"] = series.groupby("iu_ac")["heures"].max().reindex(zeros_arc.index).fillna(0).astype(int)
zeros_arc["libelle"] = zeros_arc.index.map(libelle)

print("Arcs avec le plus de q = 0 :")
display(zeros_arc.sort_values("zeros", ascending=False).head(8))
print("Arcs avec la plus longue série de q = 0 :")
display(zeros_arc.sort_values("plus_longue_serie_h", ascending=False).head(8))
print("Arcs ayant au moins une série de q = 0 de 24 h ou plus :", int((zeros_arc["plus_longue_serie_h"] >= 24).sum()))

Les zéros sont rares (0,19 % des q renseignés) et se concentrent la nuit (pic vers 2 h UTC). Sur 12 mois, de **vraies pannes apparaissent** : trois arcs de bretelles (`A3_bretelle_1`, `AI_A3_bretelle_2`, `Chapelle_bretelle_4`) ont des séries de 240 à 484 heures consécutives à zéro (10 à 20 jours) ; `AI_A3_bretelle_2` n'a même que des zéros. 44 arcs ont au moins une série de 24 h ou plus. Les séries sont coupées par les valeurs manquantes, donc les vraies durées sont au moins aussi longues.

À l'inverse, les arcs qui cumulent le plus de zéros hors bretelles (avenue Victor-Hugo : 15 % ; boulevard Malesherbes : 10 %) n'ont pas de série de plus de 12 h : ce sont des rues très calmes la nuit. Une règle vrai zéro / panne se justifie donc : un seuil de durée (par exemple 24 h consécutives à zéro) reste à choisir et à vérifier sur les 44 arcs concernés.

### 6.3 Cohérence entre `q` et `k`, valeurs élevées

In [ ]:
print("q > 0 avec k = 0           :", nb(((trafic["q"] > 0) & (trafic["k"] == 0)).sum()))
print("q = 0 avec k >= 15 %       :", nb(((trafic["q"] == 0) & (trafic["k"] >= 15)).sum()), "(rue pleine sans véhicule compté)")
print("q = 0 avec k > 5 %         :", nb(((trafic["q"] == 0) & (trafic["k"] > 5)).sum()))

par_arc_q = trafic.groupby("iu_ac")["q"].agg(["mean", "max"]).dropna()
top = par_arc_q.sort_values("mean", ascending=False).head(8).round(0)
top["libelle"] = top.index.map(libelle)
print("\nArcs au débit moyen le plus élevé (véh/h) :")
display(top)

Les incohérences q / k sont marginales (23 769 lignes avec q > 0 et k = 0, 384 avec q = 0 et k >= 15 %, sur 25,7 millions). Les débits moyens les plus élevés (4 500 à 5 000 véh/h) viennent d'arcs aux libellés `PE_*` / `PI_*` (probablement le périphérique extérieur / intérieur) : ce sont des valeurs élevées mais cohérentes avec ce type de voie, pas des erreurs.

### 6.4 Profil moyen heure × jour de semaine

Contrôle de plausibilité, en heure UTC et avant décalage « fin -> début » de période.

In [ ]:
jours = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, nom in zip(axes, ["q", "k"], ["débit moyen q (véh/h)", "taux d'occupation moyen k (%)"]):
    m = trafic.pivot_table(index=trafic["t_1h"].dt.hour, columns=trafic["t_1h"].dt.dayofweek,
                           values=col, aggfunc="mean")
    im = ax.imshow(m.values, aspect="auto", origin="lower", cmap="Blues")
    ax.set_xticks(range(7), jours)
    ax.set_yticks(range(0, 24, 3))
    ax.set_title(nom)
    ax.set_xlabel("jour de la semaine")
    ax.set_ylabel("heure UTC (fin de période)")
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("Le trafic est plus faible la nuit et le week-end : les mesures sont plausibles",
             x=0.01, ha="left", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Bilan

In [ ]:
n_vides_qk = int(((par_arc["q_nan"] == 1) & (par_arc["k_nan"] == 1)).sum())
bilan = pd.DataFrame([
    ("Référentiel", "Arcs avec plusieurs versions de géométrie",
     f"{len(ecarts)} arcs, {nb(len(ref) - len(n_versions))} lignes en trop ; écart de Hausdorff médian {ecarts['ecart_hausdorff'].median():.0f} m",
     "règle de choix de la version"),
    ("Référentiel", "Dates de validité toutes expirées avant la période",
     part((ref['date_fin'] < DEBUT).sum(), len(ref)) + " des lignes", "ignorer date_debut / date_fin"),
    ("Jointure", "Arcs du trafic sans géométrie",
     f"{len(ids_trafic - ids_ref)} arcs, {part(len(sans_ref), len(trafic))} des lignes", "exclure ou garder sans position"),
    ("Jointure", "Gonflement si le référentiel n'est pas dédoublonné",
     f"{nb(len(jointure) - len(trafic))} lignes en plus (+{len(jointure) / len(trafic) - 1:.1%})".replace(".", ","),
     "dédoublonner avant de joindre"),
    ("Trafic", "Heures absentes de la grille",
     f"{part(attendues - len(trafic), attendues)} ; dont {len(heure_vide)} heures sans aucune ligne", "reconstruire la grille complète"),
    ("Trafic", "Arcs 100 % vides (q et k)", f"{n_vides_qk} arcs sur {len(par_arc)}", "supprimer ces arcs"),
    ("Trafic", "Arcs avec q seul ou k seul", f"{int((type_arc == 'k seul').sum())} k seul, {int((type_arc == 'q seul').sum())} q seul",
     "garder, ou exiger q et k"),
    ("Trafic", "Lignes avec q ou k manquant", part(int((q_nan | k_nan).sum()), n), "NaN conservés, imputation plus tard"),
    ("Trafic", "etat_barre = Invalide", part(int((trafic['etat_barre'] == 'Invalide').sum()), n), "mettre q et k à NaN (déjà le cas)"),
    ("Trafic", "q = 0 / k = 0", f"{nb(int(zq.sum()))} / {nb(int(zk.sum()))} lignes", "règle vrai zéro / panne"),
], columns=["source", "problème", "ampleur (oct. 2025 - sept. 2026)", "décision à prendre"])
bilan.style.hide(axis="index").set_properties(**{"text-align": "left"})

### Choix à faire

1. **Référentiel** : quelle version garder pour un `iu_ac` dupliqué (par exemple la `date_fin` la plus récente) ?
2. **Arcs à conserver** : seuil de NaN et type d'arc exigé (q et k, ou q ou k). Les arcs vides sont déjà exclus de fait ; sur 12 mois le seuil change nettement le nombre d'arcs (5 % ou 50 % : de 612 à 1 402 arcs avec q et k).
3. **Arcs sans géométrie** (31, dont 21 avec des mesures) : les écarter, ou chercher leur position ailleurs ?
4. **Zéros** : fixer le seuil de durée qui sépare vrais zéros (nuit) et pannes (séries longues, bretelles).
5. **Heures absentes** : les 129 heures sans aucune ligne (dont cinq journées entières) ne se corrigent pas par arc ; comprendre pourquoi l'heure du changement d'automne manque et pas celle de mars.
6. **Arcs qui entrent ou sortent du jeu** (6 arcs) : les garder sur leur période de présence, ou les écarter.
7. Les seuils et règles retenus sont à fixer dans `config.yaml`.

## 8. Pour le plaisir : la place de l'Étoile

Mêmes données (les 12 mois). Le référentiel désigne la place par un nœud, `Pl_Charles_de_Gaulle` : on garde les arcs qui y entrent (« vers l'Étoile ») ou en sortent (« depuis l'Étoile »). Aucun arc ne décrit l'anneau de la place lui-même : on observe seulement les avenues qui y mènent. Un arc (Kléber, « depuis l'Étoile ») est absent du référentiel : il est retrouvé par le nom de ses nœuds (section 3) et compté dans les chiffres, mais n'apparaît pas sur la carte.

In [ ]:
NOEUD = "Pl_Charles_de_Gaulle"

etoile = (ref_geo.sort_values("date_fin").drop_duplicates("iu_ac", keep="last")
          .query("libelle_nd_amont == @NOEUD or libelle_nd_aval == @NOEUD").copy())
etoile["sens"] = np.where(etoile["libelle_nd_aval"] == NOEUD, "vers l'Étoile", "depuis l'Étoile")
etoile["avenue"] = etoile["libelle"].str.replace(r"^Av_(de_la_|de_|des_)?", "", regex=True).str.replace("_", " ")
etoile["nom"] = etoile["avenue"] + " (" + etoile["sens"] + ")"

# un arc de la place (Kléber, iu_ac 4443) est absent du référentiel : on le retrouve par le nom de ses nœuds (section 3)
sans_geo = sans_ref_noms.reset_index()
sans_geo = sans_geo[(sans_geo["libelle_nd_amont"] == NOEUD) | (sans_geo["libelle_nd_aval"] == NOEUD)].copy()
sans_geo["sens"] = np.where(sans_geo["libelle_nd_aval"] == NOEUD, "vers l'Étoile", "depuis l'Étoile")
sans_geo["avenue"] = sans_geo["libelle"].str.replace(r"^Av_(de_la_|de_|des_)?", "", regex=True).str.replace("_", " ")
sans_geo["nom"] = sans_geo["avenue"] + " (" + sans_geo["sens"] + ")"
tous = pd.concat([etoile.drop(columns="geometry"), sans_geo[["iu_ac", "libelle", "sens", "avenue", "nom"]]], ignore_index=True)

x = trafic[trafic["iu_ac"].isin(tous["iu_ac"])]
dispo = x.groupby("iu_ac").agg(h_q=("q", "count"), h_k=("k", "count"),
                               h_invalide=("etat_barre", lambda s: (s == "Invalide").sum()),
                               q_moyen=("q", "mean"), k_moyen=("k", "mean"))
etoile = etoile.merge(dispo, left_on="iu_ac", right_index=True, how="left")
tous = tous.merge(dispo, left_on="iu_ac", right_index=True, how="left")

sans_mesure = tous[(tous["h_q"] == 0) & (tous["h_k"] == 0)]
print(f"{len(tous)} arcs ({len(sans_geo)} sans tracé), {tous['avenue'].nunique()} avenues | sans aucune mesure : {len(sans_mesure)} "
      f"| avec k : {(tous['h_k'] > 0).sum()} | avec q : {(tous['h_q'] > 0).sum()} (sur {nb(len(heures))} heures)")
tous.sort_values("nom")[["iu_ac", "nom", "h_q", "h_k", "h_invalide", "q_moyen", "k_moyen"]].set_index("iu_ac").round(1)

In [ ]:
points = [a.geometry.coords[-1] if a["sens"] == "vers l'Étoile" else a.geometry.coords[0] for _, a in etoile.iterrows()]
cx, cy = np.mean(points, axis=0)
xmin, ymin, xmax, ymax = etoile.total_bounds
marge = 150

fig, ax = plt.subplots(figsize=(8, 7.5))
ref_geo.plot(ax=ax, color=GRIS, linewidth=1, alpha=0.5)
cmap = LinearSegmentedColormap.from_list("bleu", ["#bcd7ee", "#08306b"])
etoile.plot(ax=ax, column="k_moyen", cmap=cmap, linewidth=4, legend=True,
            legend_kwds={"label": "taux d'occupation moyen k (%)", "shrink": 0.6},
            missing_kwds={"color": ORANGE})
for avenue, g in etoile.groupby("avenue"):
    bouts = [p for geom in g.geometry for p in (geom.coords[0], geom.coords[-1])]
    loin = max(bouts, key=lambda p: (p[0] - cx) ** 2 + (p[1] - cy) ** 2)
    dx, dy = loin[0] - cx, loin[1] - cy
    n = np.hypot(dx, dy)
    ax.annotate(avenue, xy=loin, xytext=(8 * dx / n, 8 * dy / n), textcoords="offset points",
                ha="left" if dx > 0 else "right", va="center", fontsize=9)
ax.set_xlim(xmin - marge, xmax + marge)
ax.set_ylim(ymin - marge, ymax + marge)
ax.legend(handles=[Line2D([0], [0], color=ORANGE, lw=4, label="aucune mesure sur la période")],
          loc="lower left", frameon=False, fontsize=9)
ax.set_title(f"{tous['avenue'].nunique()} avenues convergent sur la place ; {len(sans_mesure)} arcs sur {len(tous)} n'ont aucune mesure")
ax.set_axis_off()
plt.show()

In [ ]:
def profil(df):
    """k moyen par heure UTC, en distinguant semaine et week-end."""
    d = df[df["k"].notna()]
    return d.groupby([d["t_1h"].dt.dayofweek >= 5, d["t_1h"].dt.hour])["k"].mean()


ordre = ["Fluide", "Pré-saturé", "Saturé", "Bloqué"]


def etats(df):
    """Part de chaque état de trafic parmi les heures où k est mesuré."""
    return df[df["k"].notna()]["etat_trafic"].value_counts(normalize=True).reindex(ordre).fillna(0)


p_etoile, p_paris = profil(x), profil(trafic)
e_etoile, e_paris = etats(x), etats(trafic)
charge_etoile, charge_paris = 1 - e_etoile["Fluide"], 1 - e_paris["Fluide"]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.5, 1]})
for p, couleur, zone in [(p_paris, GRIS, "Paris"), (p_etoile, BLEU, "Étoile")]:
    for we, style, jour in [(False, "-", "semaine"), (True, "--", "week-end")]:
        s = p.xs(we, level=0)
        a1.plot(s.index, s.values, style, color=couleur, linewidth=2, label=f"{zone}, {jour}")
a1.set_xlabel("heure UTC (fin de période)")
a1.set_ylabel("taux d'occupation moyen k (%)")
a1.set_title("L'Étoile est plus occupée que Paris, surtout la nuit et le week-end")
a1.legend(frameon=False)

couleurs_etat = ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]
for y, (nom_zone, e) in enumerate([("Paris", e_paris), ("Étoile", e_etoile)]):
    gauche = 0
    for etat, couleur in zip(ordre, couleurs_etat):
        a2.barh(y, e[etat] * 100, left=gauche, color=couleur, edgecolor="white", label=etat if y == 0 else None)
        if e[etat] > 0.04:
            a2.text(gauche + e[etat] * 50, y, f"{e[etat]:.0%}", ha="center", va="center", fontsize=8,
                    color="white" if etat != "Fluide" else "black")
        gauche += e[etat] * 100
a2.set_yticks([0, 1], ["Paris", "Étoile"])
a2.set_xlabel("part des heures mesurées (%)")
a2.set_title(f"Heures chargées (k >= 15 %) : {charge_etoile:.0%} à l'Étoile, {charge_paris:.0%} à Paris")
a2.legend(frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.45), fontsize=8)
a2.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

In [ ]:
mesures = x[x["k"].notna()].assign(chargee=lambda d: d["k"] >= 15).groupby("iu_ac")["chargee"].mean()
tab = (etoile.set_index("iu_ac").loc[mesures.index, ["nom", "sens", "q_moyen", "k_moyen"]]
       .assign(part_heures_chargees=mesures).sort_values("part_heures_chargees"))

fig, ax = plt.subplots(figsize=(8, 5))
couleur_sens = {"vers l'Étoile": BLEU, "depuis l'Étoile": ORANGE}
ax.barh(tab["nom"], tab["part_heures_chargees"] * 100, color=tab["sens"].map(couleur_sens))
ax.axvline(charge_paris * 100, color="black", linewidth=1, linestyle=":")
ax.text(charge_paris * 100 + 1, 0, "moyenne Paris", fontsize=8, va="center")
ax.set_xlabel("part des heures avec k >= 15 % (%)")
top3 = ", ".join(tab.sort_values("part_heures_chargees", ascending=False).head(3)["nom"].str.split(" \\(").str[0])
ax.set_title(f"Les heures chargées se concentrent sur : {top3}")
ax.legend(handles=[Line2D([0], [0], color=c, lw=6, label=s) for s, c in couleur_sens.items()],
          frameon=False, loc="lower right")
ax.grid(axis="y", visible=False)
plt.show()

tab.sort_values("part_heures_chargees", ascending=False).round(2)

### Ce que montrent les 12 mois

- **Couverture** : sur 22 arcs, 5 n'ont aucune mesure de l'année (Carnot et Iéna dans les deux sens, Kléber depuis l'Étoile : invalides ou absents). 17 arcs ont `k`, 13 ont `q`. Les arcs mesurés perdent en général environ 200 heures (invalides) ; les Champs-Élysées en perdent 3 100 à 3 600 (un tiers de l'année), et Wagram est presque toujours invalide vers l'Étoile (7 698 heures) tandis que son sens inverse n'a quasiment pas de `k` (4 heures).
- **Plus chargée que Paris** : 23 % des heures mesurées ont k >= 15 %, contre 13 % pour l'ensemble de Paris. L'écart existe à toute heure ; il est le plus net la nuit et le week-end.
- **Ce sont toujours les sorties qui coincent** : Marceau (74 % des heures), Friedland (72 %) et Foch (66 %) dans le sens « depuis l'Étoile ». Côté entrées, seuls Friedland (37 %) et Hoche (17 %) dépassent la moyenne de Paris. Les 12 autres arcs mesurés sont chargés moins de 12 % du temps, y compris les Champs-Élysées (6 % et 11 %). La congestion reste concentrée sur 3 à 5 arcs, pas sur toute la place. Les mêmes sorties dominaient en octobre seul, un peu plus marquées (Foch 82 %).

Limites : `k` est une occupation en un point de la voie, pas une vitesse ; l'anneau de la place n'est pas mesuré ; la cause des sorties chargées n'est pas établie (position des capteurs, feux, capacité des avenues) ; « Paris » est la moyenne de tous les arcs, pas un échantillon d'avenues comparable ; l'arc Victor Hugo vers l'Étoile a un débit moyen de 86 véh/h et un `k` moyen de 0,3 %, donc ce capteur mesure sans doute une voie très peu utilisée et ne dit rien de l'avenue ; les 12 mois mélangent été et hiver, les profils horaires ne les distinguent pas.